# Hybrid-Approach Seasonal Projection Differences — SSP2-4.5
## Notebook 12 — Compute & Save Difference NetCDF Files

**Approach:** Each basin uses either **Best Single Model** or **3-Model Ensemble** (NB11 output). No basin uses the 6-model ensemble.  
**Data sources:** NB05 Jordan-wide 3-model NC files | raw single-model NC files from `D:\\RICAAR\\...`  
**Outputs:** `comments\\projections\\seasonal_means\\` and `comments\\projections\\differences\\`

## 1. Imports

In [1]:
import warnings
import datetime
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from pathlib import Path
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore")
print("Libraries loaded.")
for lib, mod in [("numpy", np), ("pandas", pd), ("xarray", xr), ("geopandas", gpd)]:
    print(f"  {lib:<12}: {mod.__version__}")

Libraries loaded.
  numpy       : 2.1.3
  pandas      : 2.2.3
  xarray      : 2025.12.0
  geopandas   : 1.0.1


## 2. Paths & Settings

In [2]:
COMMENTS_ROOT = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments")

SHAPEFILE        = COMMENTS_ROOT / r"jordan.basins\surface.basins.for.jordan\Jo.shp"
RANKINGS_CSV     = COMMENTS_ROOT / r"validation\single.model\basin_model_rankings.csv"
RECOMMENDATION_CSV = COMMENTS_ROOT / r"validation\comparison\basin_approach_recommendation.csv"

# NB05 Jordan-wide 3-model ensemble NC files
ENS3_DIR = COMMENTS_ROOT / "3 ensemble models"
# Pattern: Jordan_3model_ensemble_ssp245_{nc_label}.nc

# Raw single-model NC files (for best-single basins)
MODEL_DIR = Path(
    r"D:\RICAAR\riccar-data_jordan-ssp2-4-5-daily-data_2024-07-29_0915"
    r"\Merge\Precipitation"
)
# Pattern: {ModelName}.nc   Variable: prAdjust (mm/day)

OUTPUT_ROOT  = COMMENTS_ROOT / "projections"
SEASONAL_DIR = OUTPUT_ROOT / "seasonal_means"
DIFF_DIR     = OUTPUT_ROOT / "differences"
SEASONAL_DIR.mkdir(parents=True, exist_ok=True)
DIFF_DIR.mkdir(parents=True, exist_ok=True)

PR_VAR       = "prAdjust"   # mm/day in all NC files
SYRIA_BASINS = {"YARMOUK (SYRIA)", "AZRAQ (SYRIA)", "AMMAN ZARQA (SYRIA)"}

# NB05 period labels (must match output filenames exactly)
PERIODS = {
    "ref" : {"nc_label": "1995_2014", "years": (1995, 2014)},
    "near": {"nc_label": "2021_2040", "years": (2021, 2040)},
    "mid" : {"nc_label": "2041_2060", "years": (2041, 2060)},
}

WET_MONTHS = [10, 11, 12, 1, 2, 3]   # Oct-Mar
DRY_MONTHS = [4, 5, 6, 7, 8, 9]      # Apr-Sep
SEASONS    = {"wet": WET_MONTHS, "dry": DRY_MONTHS}

print("Paths configured.")
print(f"  ENS3 dir  : {ENS3_DIR}")
print(f"  Model dir : {MODEL_DIR}")
print(f"  Output    : {OUTPUT_ROOT}")

Paths configured.
  ENS3 dir  : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\3 ensemble models
  Model dir : D:\RICAAR\riccar-data_jordan-ssp2-4-5-daily-data_2024-07-29_0915\Merge\Precipitation
  Output    : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\projections


## 3. Load Rankings and Approach Recommendations

Reads `basin_approach_recommendation.csv` (NB11) for the 12 validated basins.  
Column `Recommended_Approach` contains `"Best Single Model"` or `"3-Model Ensemble"` — no basin uses 6-model ensemble.

In [3]:
# ── NB03 rankings: best single model and top-3 per validated basin ────────────
rankings = pd.read_csv(RANKINGS_CSV)
rankings.columns = [c.strip() for c in rankings.columns]

top3_per_basin = (
    rankings.sort_values(["Basin", "Avg_Rank"])
    .groupby("Basin")["Model"]
    .apply(lambda x: x.head(3).tolist())
    .to_dict()
)
best_single_per_basin = (
    rankings.sort_values(["Basin", "Avg_Rank"])
    .groupby("Basin")["Model"]
    .first()
    .to_dict()
)

# ── NB11 recommendation: approach per validated basin ─────────────────────────
rec_df = pd.read_csv(RECOMMENDATION_CSV)
rec_df.columns = [c.strip() for c in rec_df.columns]

LABEL_TO_CODE = {
    "Best Single Model" : "single",
    "3-Model Ensemble"  : "ens3",
}
basin_approach = {
    row["Basin"]: LABEL_TO_CODE[row["Recommended_Approach"]]
    for _, row in rec_df.iterrows()
    if row["Recommended_Approach"] in LABEL_TO_CODE
}

print("Approach per validated basin (NB11):")
for b, a in basin_approach.items():
    extra = f"  [{best_single_per_basin.get(b)}]" if a == "single" else ""
    print(f"  {b:<30} → {a}{extra}")

Approach per validated basin (NB11):
  N.R.S.W                        → ens3
  YARMOUK (JORDAN)               → single  [MPI-ESM1-2-LR]
  JORDAN VALLY (JORDAN)          → single  [MPI-ESM1-2-LR]
  AMMAN ZARQA (JORDAN)           → single  [MPI-ESM1-2-LR]
  S.R.S.W                        → single  [MPI-ESM1-2-LR]
  D.S.R.S.W                      → ens3
  MUJIB                          → single  [MPI-ESM1-2-LR]
  W. ARABA NORTH                 → ens3
  HASA                           → single  [MPI-ESM1-2-LR]
  AZRAQ (JORDAN)                 → ens3
  JAFER                          → single  [NorESM2-MM]
  HAMMAD                         → single  [MPI-ESM1-2-LR]


## 4. Load Shapefile and Propagate Assignment to Unranked Basins via KD-Tree

In [4]:
gdf_all    = gpd.read_file(SHAPEFILE)
gdf_jordan = gdf_all[~gdf_all["BASIN_NAME"].isin(SYRIA_BASINS)].copy().reset_index(drop=True)
gdf_jordan["cx"] = gdf_jordan.geometry.centroid.x
gdf_jordan["cy"] = gdf_jordan.geometry.centroid.y

# Build KD-tree on the 12 validated (ranked) basin centroids
ranked_rows   = [r for _, r in gdf_jordan.iterrows() if r["BASIN_NAME"] in basin_approach]
ranked_coords = np.array([[r["cy"], r["cx"]] for r in ranked_rows])
ranked_names  = [r["BASIN_NAME"] for r in ranked_rows]
kd_tree       = cKDTree(ranked_coords)

# Propagate to the 4 unranked basins
for _, row in gdf_jordan.iterrows():
    b = row["BASIN_NAME"]
    if b in basin_approach:
        continue
    dist_deg, idx         = kd_tree.query([row["cy"], row["cx"]])
    nearest               = ranked_names[idx]
    basin_approach[b]     = basin_approach[nearest]
    top3_per_basin[b]     = top3_per_basin[nearest]
    best_single_per_basin[b] = best_single_per_basin[nearest]
    print(f"  KD-tree: {b:<35} → '{basin_approach[b]}'  (nearest: {nearest}, {dist_deg*111.32:.1f} km)")

print()
print(f"Total Jordan basins assigned: {len(basin_approach)}")
print()
print(f"{'Basin':<35} {'Approach':<8}  Model(s)")
print("-" * 85)
for _, row in gdf_jordan.iterrows():
    b = row["BASIN_NAME"]
    a = basin_approach[b]
    models = best_single_per_basin[b] if a == "single" else " | ".join(top3_per_basin[b])
    print(f"  {b:<33} {a:<8}  {models}")

  KD-tree: J.VALLEY-YARMOUK TRIANGLE           → 'ens3'  (nearest: N.R.S.W, 29.2 km)
  KD-tree: WADI ARABA SOUTH                    → 'ens3'  (nearest: W. ARABA NORTH, 100.5 km)
  KD-tree: QA DISI & SOUTHERN DESERT           → 'single'  (nearest: JAFER, 74.3 km)
  KD-tree: SARHAN                              → 'ens3'  (nearest: AZRAQ (JORDAN), 119.8 km)

Total Jordan basins assigned: 16

Basin                               Approach  Model(s)
-------------------------------------------------------------------------------------
  HAMMAD                            single    MPI-ESM1-2-LR
  YARMOUK (JORDAN)                  single    MPI-ESM1-2-LR
  J.VALLEY-YARMOUK TRIANGLE         ens3      NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5
  JORDAN VALLY (JORDAN)             single    MPI-ESM1-2-LR
  N.R.S.W                           ens3      NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5
  AZRAQ (JORDAN)                    ens3      MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
  AMMAN ZARQA (JORDAN)      

## 5. Load basin_id Grid from NB05 Output

NB05 embedded a `basin_id` variable (int8, shape: lat × lon) in every 3-model NC file.  
Value = basin index (0-based, matching shapefile row order); -1 = outside Jordan.  
The attribute `basin_id_labels` has format `"0=HAMMAD; 1=YARMOUK (JORDAN); ..."`

In [5]:
sample_nc  = ENS3_DIR / "Jordan_3model_ensemble_ssp245_1995_2014.nc"
ds_sample  = xr.open_dataset(sample_nc)

lats           = ds_sample["lat"].values
lons           = ds_sample["lon"].values
n_lat, n_lon   = len(lats), len(lons)
basin_id_grid  = ds_sample["basin_id"].values.astype(np.int8)  # -1 outside Jordan

# Parse basin_id_labels: format is "0=HAMMAD; 1=YARMOUK (JORDAN); ..."
labels_str  = ds_sample["basin_id"].attrs["basin_id_labels"]
idx_to_name = {}   # int → basin name
for part in labels_str.split(";"):
    part = part.strip()
    if "=" in part:
        idx_s, bname = part.split("=", 1)
        idx_to_name[int(idx_s.strip())] = bname.strip()

ds_sample.close()

print(f"Grid       : {n_lat} lat x {n_lon} lon")
print(f"Inside JO  : {int((basin_id_grid >= 0).sum())} cells")
print(f"Outside JO : {int((basin_id_grid < 0).sum())} cells")
print()
print(f"{'idx':<4} {'Basin name':<35} {'Approach':<8}  Model(s)")
print("-" * 90)
for idx in sorted(idx_to_name):
    b    = idx_to_name[idx]
    a    = basin_approach.get(b, "?")
    n    = int((basin_id_grid == idx).sum())
    mstr = best_single_per_basin.get(b, "?") if a == "single" else " | ".join(top3_per_basin.get(b, []))
    print(f"  {idx:>2}  {b:<33} {a:<8}  {mstr}  [{n} cells]")

Grid       : 85 lat x 75 lon
Inside JO  : 6375 cells
Outside JO : 0 cells

idx  Basin name                          Approach  Model(s)
------------------------------------------------------------------------------------------
   0  HAMMAD                            single    MPI-ESM1-2-LR  [5711 cells]
   1  YARMOUK (JORDAN)                  single    MPI-ESM1-2-LR  [11 cells]
   2  J.VALLEY-YARMOUK TRIANGLE         ens3      NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5  [0 cells]
   3  JORDAN VALLY (JORDAN)             single    MPI-ESM1-2-LR  [5 cells]
   4  N.R.S.W                           ens3      NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5  [11 cells]
   5  AZRAQ (JORDAN)                    ens3      MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5  [113 cells]
   6  AMMAN ZARQA (JORDAN)              single    MPI-ESM1-2-LR  [33 cells]
   7  S.R.S.W                           single    MPI-ESM1-2-LR  [6 cells]
   8  MUJIB                             single    MPI-ESM1-2-LR  [61 cells]
   9  D.S.

## 6. Helper Functions

In [6]:
def seasonal_mean_from_nc(nc_path, season_months):
    """
    Open a Jordan-wide NC (prAdjust, mm/day), compute standardised monthly
    totals (Eq. 9: P_monthly = 30/D_m * sum_daily), then return the
    long-term seasonal mean as (n_lat, n_lon) float32 [mm/month].
    """
    ds      = xr.open_dataset(nc_path)
    da      = ds[PR_VAR]                                         # (time, lat, lon)
    da_msum = da.resample(time="1MS").sum(skipna=True)           # monthly sum
    da_mstd = da_msum * (30.0 / da_msum["time"].dt.days_in_month)  # standardise
    mask    = da_mstd["time"].dt.month.isin(season_months)
    result  = da_mstd.sel(time=mask).mean(dim="time", skipna=True).values
    ds.close()
    return result.astype(np.float32)


def single_model_seasonal_mean(model_name, year_range, season_months):
    """
    Same but for a raw single-model NC file, sliced to year_range.
    """
    nc_path = MODEL_DIR / f"{model_name}.nc"
    ds      = xr.open_dataset(nc_path)
    da      = ds[PR_VAR].sel(time=slice(str(year_range[0]), str(year_range[1])))
    da_msum = da.resample(time="1MS").sum(skipna=True)
    da_mstd = da_msum * (30.0 / da_msum["time"].dt.days_in_month)
    mask    = da_mstd["time"].dt.month.isin(season_months)
    result  = da_mstd.sel(time=mask).mean(dim="time", skipna=True).values
    ds.close()
    return result.astype(np.float32)


print("Helper functions defined.")

Helper functions defined.


## 7. Build and Save Hybrid Seasonal Mean NC Files

For each (period, season):
1. Load the 3-model seasonal mean from NB05 output (covers all ens3 basins).
2. Load single-model seasonal means only for models needed by `single` basins.
3. Assemble the hybrid grid using `basin_id` lookup — no pixel outside Jordan gets a value.
4. Save as compressed NetCDF4.

In [7]:
def build_and_save_seasonal_mean(period_key, season_key):
    pcfg   = PERIODS[period_key]
    p_lbl  = pcfg["nc_label"]    # e.g. "1995_2014"
    p_yrs  = pcfg["years"]       # e.g. (1995, 2014)
    s_mths = SEASONS[season_key]

    outfile = SEASONAL_DIR / f"hybrid_ssp245_{period_key}_{season_key}.nc"
    if outfile.exists():
        print(f"  [SKIP — exists] {outfile.name}")
        return outfile

    print(f"\nBuilding: period={period_key} ({p_lbl})  |  season={season_key}")

    # ── 1. Load 3-model seasonal mean (covers all ens3 basins) ───────────────
    nc3      = ENS3_DIR / f"Jordan_3model_ensemble_ssp245_{p_lbl}.nc"
    print(f"  Loading ens3 ← {nc3.name}")
    data_ens3 = seasonal_mean_from_nc(nc3, s_mths)   # (n_lat, n_lon)

    # ── 2. Load single-model means — only models actually needed ──────────────
    needed_models = {
        best_single_per_basin[b]
        for b in basin_approach
        if basin_approach[b] == "single" and b in best_single_per_basin
    }
    data_single = {}   # model_name → (n_lat, n_lon)
    for model in sorted(needed_models):
        print(f"  Loading single ← {model}")
        data_single[model] = single_model_seasonal_mean(model, p_yrs, s_mths)

    # ── 3. Assemble hybrid grid ───────────────────────────────────────────────
    print("  Assembling hybrid grid ...")
    hybrid = np.full((n_lat, n_lon), np.nan, dtype=np.float32)

    for i in range(n_lat):
        for j in range(n_lon):
            cell_id = int(basin_id_grid[i, j])
            if cell_id < 0:
                continue                             # outside Jordan → stays NaN

            bname    = idx_to_name.get(cell_id)
            approach = basin_approach.get(bname, "ens3")   # default ens3 if unknown

            if approach == "single":
                model = best_single_per_basin.get(bname)
                if model and model in data_single:
                    hybrid[i, j] = data_single[model][i, j]
                else:
                    hybrid[i, j] = data_ens3[i, j]  # safe fallback
            else:   # ens3
                hybrid[i, j] = data_ens3[i, j]

    # ── 4. Save ───────────────────────────────────────────────────────────────
    ds_out = xr.Dataset(
        {"pr": xr.DataArray(
            hybrid, dims=["lat", "lon"],
            attrs={
                "long_name"    : "Hybrid-approach seasonal mean (30-day standardised)",
                "units"        : "mm/month",
                "season"       : season_key,
                "season_months": str(s_mths),
                "period"       : f"{p_yrs[0]}-{p_yrs[1]}",
                "scenario"     : "SSP2-4.5",
                "approach"     : "hybrid — per-basin best approach from NB11",
            }
        )},
        coords={
            "lat": ("lat", lats, {"units": "degrees_north", "long_name": "latitude",  "axis": "Y"}),
            "lon": ("lon", lons, {"units": "degrees_east",  "long_name": "longitude", "axis": "X"}),
        },
        attrs={
            "title"      : f"Hybrid seasonal mean SSP2-4.5 {period_key} {season_key}",
            "Conventions": "CF-1.7",
            "created"    : str(datetime.datetime.now()),
            "source"     : "RICAAR SMHI-HCLIM-ALADIN-38, SMHI-MIdAS021",
        }
    )
    encoding = {"pr": {"zlib": True, "complevel": 4, "dtype": "float32",
                       "_FillValue": np.float32(1e+20)}}
    ds_out.to_netcdf(outfile, encoding=encoding, format="NETCDF4")

    v = hybrid[~np.isnan(hybrid)]
    print(f"  Saved → {outfile.name}")
    print(f"    cells={v.size}  mean={v.mean():.2f}  max={v.max():.2f}  min={v.min():.2f} mm/month")
    return outfile


print("build_and_save_seasonal_mean() defined.")

build_and_save_seasonal_mean() defined.


## 8. Run — Build All 6 Seasonal Mean Files

In [8]:
seasonal_files = {}   # (period_key, season_key) → Path

for period_key in ["ref", "near", "mid"]:
    for season_key in ["wet", "dry"]:
        path = build_and_save_seasonal_mean(period_key, season_key)
        seasonal_files[(period_key, season_key)] = path

print("\n" + "=" * 55)
print("All 6 seasonal mean files complete.")
print("=" * 55)


Building: period=ref (1995_2014)  |  season=wet
  Loading ens3 ← Jordan_3model_ensemble_ssp245_1995_2014.nc
  Loading single ← MPI-ESM1-2-LR
  Loading single ← NorESM2-MM
  Assembling hybrid grid ...
  Saved → hybrid_ssp245_ref_wet.nc
    cells=6375  mean=13.62  max=138.21  min=0.00 mm/month

Building: period=ref (1995_2014)  |  season=dry
  Loading ens3 ← Jordan_3model_ensemble_ssp245_1995_2014.nc
  Loading single ← MPI-ESM1-2-LR
  Loading single ← NorESM2-MM
  Assembling hybrid grid ...
  Saved → hybrid_ssp245_ref_dry.nc
    cells=6375  mean=2.13  max=22.20  min=0.00 mm/month

Building: period=near (2021_2040)  |  season=wet
  Loading ens3 ← Jordan_3model_ensemble_ssp245_2021_2040.nc
  Loading single ← MPI-ESM1-2-LR
  Loading single ← NorESM2-MM
  Assembling hybrid grid ...
  Saved → hybrid_ssp245_near_wet.nc
    cells=6375  mean=14.83  max=138.09  min=0.00 mm/month

Building: period=near (2021_2040)  |  season=dry
  Loading ens3 ← Jordan_3model_ensemble_ssp245_2021_2040.nc
  Loadin

## 9. Compute Differences (Future − Reference) and Save

In [9]:
PERIOD_TITLES = {"near": "2021-2040", "mid": "2041-2060"}

# Load reference grids once
ref_grids = {}
for season_key in ["wet", "dry"]:
    ds = xr.open_dataset(seasonal_files[("ref", season_key)])
    ref_grids[season_key] = ds["pr"].values.astype(np.float32)
    ds.close()

diff_files = {}

for future_period in ["near", "mid"]:
    for season_key in ["wet", "dry"]:
        outfile = DIFF_DIR / f"hybrid_ssp245_{future_period}_{season_key}_diff.nc"

        if outfile.exists():
            print(f"[SKIP — exists] {outfile.name}")
            diff_files[(future_period, season_key)] = outfile
            continue

        ds_fut = xr.open_dataset(seasonal_files[(future_period, season_key)])
        pr_fut = ds_fut["pr"].values.astype(np.float32)
        pr_ref = ref_grids[season_key]
        delta  = pr_fut - pr_ref
        ds_fut.close()

        ds_diff = xr.Dataset(
            {
                "pr_diff": xr.DataArray(
                    delta, dims=["lat", "lon"],
                    attrs={
                        "long_name"    : f"Change in monthly precip: {PERIOD_TITLES[future_period]} minus 1995-2014",
                        "units"        : "mm/month",
                        "scenario"     : "SSP2-4.5",
                        "approach"     : "hybrid",
                        "future_period": PERIOD_TITLES[future_period],
                        "reference"    : "1995-2014",
                        "season"       : season_key,
                    }
                ),
                "pr_ref": xr.DataArray(
                    pr_ref, dims=["lat", "lon"],
                    attrs={"long_name": "Reference (1995-2014) seasonal mean",
                           "units": "mm/month"}
                ),
                "pr_future": xr.DataArray(
                    pr_fut, dims=["lat", "lon"],
                    attrs={"long_name": f"{PERIOD_TITLES[future_period]} seasonal mean",
                           "units": "mm/month"}
                ),
            },
            coords={
                "lat": ("lat", lats, {"units": "degrees_north", "long_name": "latitude",  "axis": "Y"}),
                "lon": ("lon", lons, {"units": "degrees_east",  "long_name": "longitude", "axis": "X"}),
            },
            attrs={
                "title"   : f"Hybrid precip change SSP2-4.5 {future_period} {season_key}",
                "Conventions": "CF-1.7",
                "created" : str(datetime.datetime.now()),
            }
        )

        enc = {v: {"zlib": True, "complevel": 4, "dtype": "float32",
                   "_FillValue": np.float32(1e+20)}
               for v in ["pr_diff", "pr_ref", "pr_future"]}
        ds_diff.to_netcdf(outfile, encoding=enc, format="NETCDF4")
        diff_files[(future_period, season_key)] = outfile

        v = delta[~np.isnan(delta)]
        print(f"{outfile.name}")
        print(f"  mean={v.mean():.2f}  max={v.max():.2f}  min={v.min():.2f}  points={v.size}")

print("\nAll 4 difference NC files saved.")

hybrid_ssp245_near_wet_diff.nc
  mean=1.21  max=10.41  min=-14.52  points=6375
hybrid_ssp245_near_dry_diff.nc
  mean=0.37  max=4.71  min=-2.88  points=6375
hybrid_ssp245_mid_wet_diff.nc
  mean=0.11  max=9.09  min=-18.91  points=6375
hybrid_ssp245_mid_dry_diff.nc
  mean=-0.25  max=2.16  min=-3.87  points=6375

All 4 difference NC files saved.


## 10. Verify All Outputs

In [10]:
print("=" * 72)
print("SEASONAL MEAN FILES  →  projections/seasonal_means/")
print("=" * 72)
for f in sorted(SEASONAL_DIR.glob("hybrid_ssp245_*.nc")):
    ds  = xr.open_dataset(f)
    pr  = ds["pr"].values
    v   = pr[~np.isnan(pr)]
    sz  = f.stat().st_size / 1024
    print(f"  {f.name:<46} cells={v.size:>3} "
          f" mean={v.mean():>6.2f}  max={v.max():>6.2f}  min={v.min():>6.2f} mm/mo  [{sz:.0f} KB]")
    ds.close()

print()
print("=" * 72)
print("DIFFERENCE FILES     →  projections/differences/")
print("=" * 72)
for f in sorted(DIFF_DIR.glob("hybrid_ssp245_*.nc")):
    ds  = xr.open_dataset(f)
    d   = ds["pr_diff"].values
    v   = d[~np.isnan(d)]
    sz  = f.stat().st_size / 1024
    print(f"  {f.name:<54} mean={v.mean():>6.2f}  max={v.max():>6.2f}  min={v.min():>6.2f}  [{sz:.0f} KB]")
    ds.close()

print()
print("Ready for Notebook 13 — Projection Maps.")

SEASONAL MEAN FILES  →  projections/seasonal_means/
  hybrid_ssp245_mid_dry.nc                       cells=6375  mean=  1.88  max= 19.95  min=  0.00 mm/mo  [34 KB]
  hybrid_ssp245_mid_wet.nc                       cells=6375  mean= 13.73  max=133.98  min=  0.00 mm/mo  [34 KB]
  hybrid_ssp245_near_dry.nc                      cells=6375  mean=  2.50  max= 20.36  min=  0.00 mm/mo  [34 KB]
  hybrid_ssp245_near_wet.nc                      cells=6375  mean= 14.83  max=138.09  min=  0.00 mm/mo  [34 KB]
  hybrid_ssp245_ref_dry.nc                       cells=6375  mean=  2.13  max= 22.20  min=  0.00 mm/mo  [34 KB]
  hybrid_ssp245_ref_wet.nc                       cells=6375  mean= 13.62  max=138.21  min=  0.00 mm/mo  [34 KB]

DIFFERENCE FILES     →  projections/differences/
  hybrid_ssp245_mid_dry_diff.nc                          mean= -0.25  max=  2.16  min= -3.87  [77 KB]
  hybrid_ssp245_mid_wet_diff.nc                          mean=  0.11  max=  9.09  min=-18.91  [77 KB]
  hybrid_ssp245_near_d